## Step 1: Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import random
import json

In [2]:
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)

PyTorch version: 2.9.0+cpu
NumPy version: 2.4.6


## Step 2: Downloading and Loading the Dataset

In [3]:
from pathlib import Path
import hashlib
import zipfile
import urllib.request

DATA_URL = "https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip"

ZIP_PATH = Path("data.zip")
DATA_DIR = Path("data")

EXPECTED_SHA256 = "9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67"

# Download the dataset if it does not exist
if not ZIP_PATH.exists():
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

# Verify the downloaded file
sha256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print("SHA-256:", sha256)

# Check whether the checksum matches
if sha256 != EXPECTED_SHA256:
    ZIP_PATH.unlink()
    raise ValueError("Incorrect download. Please run this cell again.")

# Extract the dataset
if not DATA_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

print("Dataset extracted successfully!")

SHA-256: 9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67
Dataset extracted successfully!


## Step 3: Loading the Dataset

In [4]:
train_dataset = datasets.ImageFolder("data/data/train")
evaluation_dataset = datasets.ImageFolder("data/data/test")

print("Training images:", len(train_dataset))
print("Evaluation images:", len(evaluation_dataset))
print("Classes:", train_dataset.classes)
print("Class mapping:", train_dataset.class_to_idx)

Training images: 800
Evaluation images: 201
Classes: ['curly', 'straight']
Class mapping: {'curly': 0, 'straight': 1}


## Step 4: Setting Random Seeds

In [5]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

## Step 5: Image Transformations

In [6]:
train_transform = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [7]:
train_dataset = datasets.ImageFolder(
    "data/data/train",
    transform=train_transform
)

evaluation_dataset = datasets.ImageFolder(
    "data/data/test",
    transform=evaluation_transform
)

print("Training images:", len(train_dataset))
print("Evaluation images:", len(evaluation_dataset))

Training images: 800
Evaluation images: 201


## Step 7: Creating DataLoaders

In [8]:
loader_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator
)

evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0
)

In [9]:
print("Training batches:", len(train_loader))
print("Evaluation batches:", len(evaluation_loader))

Training batches: 40
Evaluation batches: 11


## Step 8: Building the Neural Network

In [10]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Conv2d(3, 32, kernel_size=3)
        self.pool = nn.MaxPool2d(2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)


model = HairModel()
model

HairModel(
  (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (hidden): Linear(in_features=313632, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=1, bias=True)
)

## Question 1: Loss Function

The correct loss function is `nn.BCEWithLogitsLoss()` because our model produces one logit for a binary classification problem.

In [11]:
criterion = nn.BCEWithLogitsLoss()
print(criterion)

BCEWithLogitsLoss()


## Question 2: Number of Trainable Parameters

Calculate the total number of trainable parameters in the neural network, including both weights and biases.

We use `sum(p.numel() for p in model.parameters())` to count all the trainable parameters in the model.

In [12]:
sum(p.numel() for p in model.parameters())

20073473

## Step 9: Configuring the Optimizer

We use Stochastic Gradient Descent (SGD) to update the model's parameters.

The learning rate is 0.002 and the momentum is 0.8, as specified in the assignment.

In [13]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8
)

## Step 10: Training the Model

The training function passes images through the neural network, calculates the prediction error (loss), and updates the model's parameters to improve its predictions.

In [14]:
def train_one_epoch(model, train_loader, criterion, optimizer):

    model.train()

    total_loss = 0
    total_correct = 0

    for images, labels in train_loader:

        labels = labels.float().view(-1, 1)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        total_loss += loss.item() * len(labels)
        total_correct += (predictions == labels).sum().item()

    average_loss = total_loss / len(train_loader.dataset)
    accuracy = total_correct / len(train_loader.dataset)

    return average_loss, accuracy

## Step 11: Evaluating the Model

The evaluation function measures the model's performance on images it does not train on. It does not update the model's parameters.

In [15]:
def evaluate(model, evaluation_loader, criterion):

    model.eval()

    total_loss = 0
    total_correct = 0

    with torch.no_grad():

        for images, labels in evaluation_loader:

            labels = labels.float().view(-1, 1)

            outputs = model(images)

            loss = criterion(outputs, labels)

            predictions = (torch.sigmoid(outputs) >= 0.5).float()

            total_loss += loss.item() * len(labels)
            total_correct += (predictions == labels).sum().item()

    average_loss = total_loss / len(evaluation_loader.dataset)
    accuracy = total_correct / len(evaluation_loader.dataset)

    return average_loss, accuracy

## Step 12: Baseline Training

Train the neural network for 10 epochs using the original training images. After each epoch, evaluate its performance on the held-out evaluation dataset.

Record training loss, training accuracy, evaluation loss, and evaluation accuracy.

In [16]:
history_baseline = {
    "train_loss": [],
    "train_accuracy": [],
    "evaluation_loss": [],
    "evaluation_accuracy": []
}

for epoch in range(10):

    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, criterion, optimizer
    )

    evaluation_loss, evaluation_accuracy = evaluate(
        model, evaluation_loader, criterion
    )

    history_baseline["train_loss"].append(train_loss)
    history_baseline["train_accuracy"].append(train_accuracy)
    history_baseline["evaluation_loss"].append(evaluation_loss)
    history_baseline["evaluation_accuracy"].append(evaluation_accuracy)

    print(
        f"Epoch {epoch + 1}: "
        f"Train Loss: {train_loss:.4f}, "
        f"Train Accuracy: {train_accuracy:.4f}, "
        f"Evaluation Loss: {evaluation_loss:.4f}, "
        f"Evaluation Accuracy: {evaluation_accuracy:.4f}"
    )

Epoch 1: Train Loss: 0.6385, Train Accuracy: 0.6450, Evaluation Loss: 0.6010, Evaluation Accuracy: 0.6567
Epoch 2: Train Loss: 0.5441, Train Accuracy: 0.7175, Evaluation Loss: 0.6003, Evaluation Accuracy: 0.6517
Epoch 3: Train Loss: 0.4955, Train Accuracy: 0.7550, Evaluation Loss: 0.6405, Evaluation Accuracy: 0.6219
Epoch 4: Train Loss: 0.4330, Train Accuracy: 0.7913, Evaluation Loss: 0.6274, Evaluation Accuracy: 0.6915
Epoch 5: Train Loss: 0.4048, Train Accuracy: 0.8087, Evaluation Loss: 0.6293, Evaluation Accuracy: 0.6617
Epoch 6: Train Loss: 0.3953, Train Accuracy: 0.8075, Evaluation Loss: 0.6332, Evaluation Accuracy: 0.7114
Epoch 7: Train Loss: 0.3437, Train Accuracy: 0.8363, Evaluation Loss: 0.6895, Evaluation Accuracy: 0.6716
Epoch 8: Train Loss: 0.2806, Train Accuracy: 0.8850, Evaluation Loss: 0.6497, Evaluation Accuracy: 0.7214
Epoch 9: Train Loss: 0.2236, Train Accuracy: 0.9100, Evaluation Loss: 0.7114, Evaluation Accuracy: 0.7164
Epoch 10: Train Loss: 0.1584, Train Accuracy: 

In [17]:
with open("history_baseline.json", "w") as f:
    json.dump(history_baseline, f, indent=4)

print("Baseline history saved successfully!")

Baseline history saved successfully!


## Question 3: Baseline Accuracy

Calculate the median of the training accuracy values recorded over the 10 baseline epochs.

In [18]:
np.median(history_baseline["train_accuracy"])

np.float64(0.808125)

## Question 4: Baseline Loss

Calculate the population standard deviation of the training loss values over the 10 baseline epochs.

In [19]:
np.std(history_baseline["train_loss"], ddof=0)

np.float64(0.13934079361796453)

## Step 15: Data Augmentation

Data augmentation creates modified versions of training images through rotation, cropping and flipping. This helps the model generalise better to images it has not seen before.

In [20]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.RandomRotation(50),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## Step 16: Loading the augmented training images
We create a new training dataset with the augmented transformations.

In [21]:
train_dataset_augmented = datasets.ImageFolder(
    "data/data/train",
    transform=train_transform_augmented
)

In [22]:
train_loader_augmented = DataLoader(
    train_dataset_augmented,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator
)

## Step 17: Training with Data Augmentation

Continue training the existing model for 10 additional epochs using augmented training images. Evaluate the model using the original, unchanged evaluation dataset.

In [23]:
history_augmented = {
    "train_loss": [],
    "train_accuracy": [],
    "evaluation_loss": [],
    "evaluation_accuracy": []
}

for epoch in range(10):

    train_loss, train_accuracy = train_one_epoch(
        model, train_loader_augmented, criterion, optimizer
    )

    evaluation_loss, evaluation_accuracy = evaluate(
        model, evaluation_loader, criterion
    )

    history_augmented["train_loss"].append(train_loss)
    history_augmented["train_accuracy"].append(train_accuracy)
    history_augmented["evaluation_loss"].append(evaluation_loss)
    history_augmented["evaluation_accuracy"].append(evaluation_accuracy)

    print(
        f"Epoch {epoch + 1}: "
        f"Train Loss: {train_loss:.4f}, "
        f"Train Accuracy: {train_accuracy:.4f}, "
        f"Evaluation Loss: {evaluation_loss:.4f}, "
        f"Evaluation Accuracy: {evaluation_accuracy:.4f}"
    )

Epoch 1: Train Loss: 0.6444, Train Accuracy: 0.6400, Evaluation Loss: 0.6283, Evaluation Accuracy: 0.6866
Epoch 2: Train Loss: 0.5865, Train Accuracy: 0.6787, Evaluation Loss: 0.7047, Evaluation Accuracy: 0.6766
Epoch 3: Train Loss: 0.5940, Train Accuracy: 0.6613, Evaluation Loss: 0.5812, Evaluation Accuracy: 0.7114
Epoch 4: Train Loss: 0.5469, Train Accuracy: 0.7150, Evaluation Loss: 0.6194, Evaluation Accuracy: 0.6816
Epoch 5: Train Loss: 0.5469, Train Accuracy: 0.7113, Evaluation Loss: 0.6176, Evaluation Accuracy: 0.7065
Epoch 6: Train Loss: 0.5207, Train Accuracy: 0.7275, Evaluation Loss: 0.5709, Evaluation Accuracy: 0.7065
Epoch 7: Train Loss: 0.5528, Train Accuracy: 0.7037, Evaluation Loss: 0.5715, Evaluation Accuracy: 0.7114
Epoch 8: Train Loss: 0.5313, Train Accuracy: 0.7250, Evaluation Loss: 0.6018, Evaluation Accuracy: 0.6965
Epoch 9: Train Loss: 0.5075, Train Accuracy: 0.7450, Evaluation Loss: 0.6582, Evaluation Accuracy: 0.6517
Epoch 10: Train Loss: 0.4897, Train Accuracy: 

## Step 18: Saving Augmented Training Results

Saving the training and evaluation metrics from the 10 augmented epochs into a JSON file.

In [24]:
with open("history_augmented.json", "w") as f:
    json.dump(history_augmented, f, indent=4)

print("Augmented history saved successfully!")

Augmented history saved successfully!


## Question 5: Mean Augmented Evaluation Loss

Calculate the average evaluation loss across the 10 augmented training epochs.

In [25]:
np.mean(history_augmented["evaluation_loss"])

np.float64(0.6091237926067998)

## Question 6: Mean Augmented Evaluation Accuracy

Calculate the average evaluation accuracy for the final five augmented epochs (epochs 6 through 10).

In [26]:
np.mean(history_augmented["evaluation_accuracy"][-5:])

np.float64(0.699502487562189)